In [ ]:
import pandas as pd
import math
import string
import os
import copy
import numpy as np

# Ballot importer function
def find_election_information(file: str, method: int = 0):
	# Read file
	if file.endswith('.csv'):
		ballot_info = pd.read_csv(filepath_or_buffer=file, header=None, usecols=[0])
		lines = [str(l).strip() for l in ballot_info.iloc[:, 0].to_list() if str(l).strip()]
	else:  # assume it's a BLT file
		with open(file, "r") as f:
			lines = [line.strip() for line in f if line.strip() and not line.startswith("#")]
	# First row = candidates, winners
	num_candidates, num_winners = map(int, lines[0].split())
	ballot_lines = lines[1:lines.index("0")]
	title = lines[-1]

	# Candidate labels: 1 → A, 2 → B, etc.
	candidate_labels = {i+1: string.ascii_uppercase[i] for i in range(num_candidates)}

	# Parse ballots
	parsed_ballots = []
	for line in ballot_lines:
		parts = list(map(int, line.split()))
		count = parts[0]
		# 0 marks the end of a ballot
		ranks = "".join(candidate_labels[int(r)] for r in parts[1:] if int(r) != 0)
		parsed_ballots.append({"count": count, "ballot": ranks})

	if method == 4: # weight check method so i have to explode each individual ballot
		ballots_to_use = []
		for ballot in parsed_ballots:
			ballots_to_use.extend([{"count": 1, "ballot": ballot["ballot"]}] * ballot["count"])
	else:
		ballots_to_use = parsed_ballots
	# DataFrame
	ballot_df = pd.DataFrame(ballots_to_use, columns=["count", "ballot"])

	return title, num_candidates, num_winners, ballot_df, candidate_labels

# Example usage
#file = "Scotland data, LEAP party information/glasgow22/Ward2.blt"
#title, num_candidates, num_winners, ballot_dataframe, candidates = find_election_information(file)

def truncate(number, digits) -> float:
	stepper = 10.0 ** digits
	return math.trunc(stepper * number) / stepper


def Scottish_STV(frame, n, S, method=0, raw_counts:bool = False, weight_threshold:float = 0):
	"""frame is a dataframe with columns 'count' and 'ballot', n is number of candidates, S is number of seats, method is exhaustion method"""
	# Define lists
	winners=set()
	hopefuls = set(string.ascii_uppercase[0:n])

	# Turn count into float and ensure numeric types
	frame['count'] = pd.to_numeric(frame['count'], errors='coerce').fillna(0.0).astype(float)
	frame['original_count'] = frame['count'].copy()

	# Set up the original frame for exhaustion tracking
	frame['already_counted'] = False
	original_frame=frame.copy(deep=True)

	cand_dict={i: string.ascii_uppercase[i] for i in range(n)}
	quota=math.floor(sum(frame['count'])/(S+1))+1
	current_round = 1

	# Key exhaustion tracking statistics (EN's research focus)
	exhausted_by_round = {1.0: 0.0}  # round number: exhausted count


	def tabulate(frame:pd.DataFrame, n: int):
		"""Iterates over frame and counts first choice votes. Returns dictionary of vote counts and original frame."""
		vote_counts = {cand_dict[i]: 0.0 for i in range(n)}

		# mask valid (non-empty, non-null) ballots
		mask = frame['ballot'].notna() & (frame['ballot'] != '')

		if mask.any():
			# extract first preference letter for each valid ballot
			first_choices = frame.loc[mask, 'ballot'].str[0]
			# sum counts by first choice
			sums = frame.loc[mask].assign(first=first_choices).groupby('first', sort=False)['count'].sum()
			# update vote_counts dict with the grouped sums
			for letter, val in sums.items():
				if letter in vote_counts:
					vote_counts[letter] = float(val) # type: ignore
				else:
					# ignore unexpected labels (preserve existing behaviour)
					continue

		return vote_counts, frame

	def remove_and_exhaust(current_round, frame:pd.DataFrame, cand, transfer_weight:float=1.0, elected:bool=False):
		"""Removes candidate from all ballots in frame. If elected=True, also transfers votes with weight.
		Preserves original behavior: returns (frame, current_round, election_over) and updates exhausted_by_round.
		"""
		election_over = False
		exhausted_this_round = float(0.0)
		current_round += 1

		# Collect indices to drop after we finish iterating
		indices_to_drop = []

		# For case 2/3 we only need to run the original_frame mask once per call.
		mask_case2_done = False
		mask_case3_done = False

		# iterate over actual index labels
		for k in list(frame.index):
			# read values once
			ballot = str(frame.at[k, 'ballot']) if 'ballot' in frame.columns else ''
			count_val = float(frame.at[k, 'count']) if 'count' in frame.columns else 0.0  # type: ignore

			# Remove candidate from ballot (if present)
			if cand and ballot != '':
				if elected and ballot[0] == cand:
					# transfer the ballot's count by weight
					frame.at[k, 'count'] = truncate(count_val * transfer_weight, 5)
					# remove any already-elected winners from this ballot
					b = ballot
					for winner in list(winners):
						if winner in b:
							b = b.replace(winner, '')
					frame.at[k, 'ballot'] = b
					ballot = b  # update local copy
				elif cand in ballot:
					new_b = ballot.replace(cand, '')
					frame.at[k, 'ballot'] = new_b
					ballot = new_b

			# Exhaustion checks per method
			match method:
				case 0:
					# traditional/final-round style: if ballot empty, exhausted
					if ballot == '':
						if not raw_counts:
							exhausted_this_round += float(frame.at[k, 'count'])  # type: ignore
						else:
							exhausted_this_round += float(frame.at[k, 'original_count'])  # type: ignore
						indices_to_drop.append(k)

				case 1:
					# same as case 0 for empty ballots
					if ballot == '':
						if not raw_counts:
							exhausted_this_round += float(frame.at[k, 'count'])  # type: ignore
						else:
							exhausted_this_round += float(frame.at[k, 'original_count'])  # type: ignore
						indices_to_drop.append(k)

					# cutting final rounds: evaluate once (uses vote_counts from outer scope)
					if not mask_case2_done:
						sorted_counts = sorted(vote_counts.values(), reverse=True)
						idx = S - len(winners) - 1
						votes_to_beat = sorted_counts[idx] if 0 <= idx < len(sorted_counts) else 0
						possible_sum = sum(vote for vote in vote_counts.values() if vote < votes_to_beat)
						if possible_sum <= votes_to_beat:
							election_over = True
							leading_hopefuls = [k_ for k_, v in vote_counts.items() if v >= votes_to_beat and k_ in hopefuls]
							winners.update(leading_hopefuls)
							print("election over due to no one being able to catch up")
							for h in leading_hopefuls:
								hopefuls.discard(h)
						mask_case2_done = True
						if election_over:
							break

				case 2:
					# first-choice representation: run mask once per call
					if not mask_case2_done:
						mask = (
							original_frame["ballot"].str[0].isin(winners)
							& (~original_frame["already_counted"])
							& original_frame["ballot"].notna()
						)
						exhausted_this_round += original_frame.loc[mask, "count"].sum()
						original_frame.loc[mask, "already_counted"] = True
						mask_case2_done = True

				case 3:
					# any winner in top S: run mask once per call
					if not mask_case3_done:
						mask = (
							original_frame["ballot"].notna()
							& (~original_frame["already_counted"])
							& original_frame["ballot"].apply(lambda b: bool(set(b[:n]) & winners))
						)
						exhausted_this_round += original_frame.loc[mask, "count"].sum()
						original_frame.loc[mask, "already_counted"] = True
						mask_case3_done = True

				case 4:
					# weight-check exhaustion: mark ballots with low weight once
					already = frame.at[k, 'already_counted'] if 'already_counted' in frame.columns else False
					if float(frame.at[k, 'count']) <= weight_threshold and not already:  # type: ignore
						exhausted_this_round += float(frame.at[k, 'original_count'])  # type: ignore[operator]
						frame.at[k, 'already_counted'] = True
						if ballot == '':
							indices_to_drop.append(k)

				case _:
					raise ValueError("Method not recognized")

		# Drop exhausted rows after iterating
		if indices_to_drop:
			frame.drop(index=indices_to_drop, inplace=True)

		# Record per-round exhausted (only when election not forced over by method logic)
		if not election_over:
			exhausted_this_round = truncate(exhausted_this_round, 5)
			exhausted_by_round[current_round] = exhausted_this_round

		return frame, current_round, election_over
	
	#print("quota ",quota)
	#print(frame)
	#print(cand_dict)
	#print(hopefuls)

	#Get each candidate's initial number of votes this round
	vote_counts, frame = tabulate(frame, n)

	for cand in hopefuls:
		if vote_counts[cand]==0:
			hopefuls.remove(cand)
	#print(vote_counts)
	#print('\n')
	max_count=max(vote_counts.values())
	while len(winners)<S:
		
		max_count=max(vote_counts.values())
		#somebody is elected and we have to transfer their votes
		if max_count>=quota:
			#There might be multiple people elected this round; save them as a sorted dictionary
			votes_for_winners={k:vote_counts[k] for k in vote_counts.keys() if vote_counts[k]>=quota}
			votes_for_winners=dict(sorted(votes_for_winners.items(),key=lambda x: x[1], reverse=True))

			# If our last winner exceeds quota
			#If we try to elect too many people, need to drop someone who surpassed quota
			if len(winners)+len(votes_for_winners)>=S:
				remaining_slots = S - len(winners)

				for cand in list(votes_for_winners.keys())[:remaining_slots]:
					winners.add(cand)
					hopefuls.discard(cand)

					if len(winners) == S:
						return winners, exhausted_by_round

			else:
				# Since we can elect all who surpassed quota, do it
				winners.update(votes_for_winners.keys())
				for cand in winners:
					if cand in hopefuls:
						hopefuls.remove(cand)
				#print(winners)

				while len(votes_for_winners)>0:
					# Pull them off and add them to winners
					cand=list(votes_for_winners.keys())[0]
					
					# Double-check
					if cand not in winners:
						winners.add(cand)
						hopefuls.remove(cand)

					if len(winners)==S:
						return winners, exhausted_by_round
					#print("cand elected", cand)
					#print('surplus',vote_counts[cand]-quota)
					
					weight=truncate((vote_counts[cand]-quota)/vote_counts[cand],5)
					frame, current_round, election_over = remove_and_exhaust(current_round, frame, cand, weight, elected=True)
					if election_over: 
						return winners, exhausted_by_round
					#print('weight',weight)
					votes_for_winners.pop(cand)
					
					# Retabulate
					vote_counts, frame = tabulate(frame, n)

					votes_for_winners={k:vote_counts[k] for k in vote_counts.keys() if vote_counts[k]>=quota }
					votes_for_winners=dict(sorted(votes_for_winners.items(),key=lambda x: x[1], reverse=True))
					#print(vote_counts)
					for cand in votes_for_winners.keys():
						if cand not in winners:
							winners.add(cand)
							hopefuls.remove(cand)
					
					if len(winners)==S:
						
						return winners, exhausted_by_round
					if method != 4:
						frame = frame.groupby("ballot", as_index=False).agg(count=("count", "sum"), original_count=("original_count", "sum"))
		#nobody is elected by surpassing quota, but the number
		#of candidates left equals S
		elif len(hopefuls)+len(winners)==S:
			# print("election over due to hopefuls + winners = S")
			winners.update(hopefuls)
			hopefuls.clear()
			frame, current_round, election_over = remove_and_exhaust(current_round, frame, '')
			if exhausted_by_round[max(exhausted_by_round)] == 0:
				del exhausted_by_round[max(exhausted_by_round)]
			return winners, exhausted_by_round
		#remove weakest cand and transfer their votes with weight one
		else:
			#print(vote_counts)
			nonzero_votes = [i for i in vote_counts.values() if i > 0]
			if not nonzero_votes:
				#print("All remaining candidates have zero votes. Ending election.")
				winners.update(hopefuls)
				return winners, exhausted_by_round
			min_count = min(nonzero_votes)
			count=0
			for votes in vote_counts.values():
				if votes==min_count:
					count+=1

			if count==1:
				eliminated_cand = str(list(vote_counts.keys())[list(vote_counts.values()).index(min_count)])
				#print("eliminate cand ",eliminated_cand)
				hopefuls.remove(eliminated_cand)
				frame, current_round, election_over = remove_and_exhaust(current_round, frame, eliminated_cand)
				if election_over:
					return winners, exhausted_by_round
				#print(hopefuls)
				vote_counts, frame = tabulate(frame, n)

				#print(vote_counts)
				#print('\n')
				max_count=max(vote_counts.values())
				if method != 4: # speeds up when i don't need weights
					frame = frame.groupby("ballot", as_index=False).agg(count=("count", "sum"), original_count=("original_count", "sum"))
				#print(frame)
			elif len(hopefuls)-count>=S-len(winners):
				eliminated_cands=[]
				for cand in vote_counts:
					if vote_counts[cand]==min_count:
						eliminated_cands.append(cand)
				#print('eliminated candidates', eliminated_cands)
				for cand in eliminated_cands:
					#print("eliminate cand ", cand)
					hopefuls.remove(cand)
					frame, current_round, election_over = remove_and_exhaust(current_round, frame, cand)
					if election_over:
						return winners, exhausted_by_round
					
				vote_counts, frame = tabulate(frame, n)
				#print(hopefuls)
				#print(vote_counts)
				#print('\n')
				max_count=max(vote_counts.values())
				if method != 4: # speeds up when i don't need weights
					frame = frame.groupby("ballot", as_index=False).agg(count=("count", "sum"), original_count=("original_count", "sum"))
				

			else:
				#print('tie',count)
				#print(original_frame)
				return set(['tie']), exhausted_by_round
	return winners, exhausted_by_round

def ballot_predictor(frame:pd.DataFrame, length_to_fill_to:int, threshold:int=5) -> pd.DataFrame:
	"""Predict the next candidate on a ballot based on existing ballots.
	'frame' is a df with 'ballot' and 'count' columns.
	'length_to_fill_to' is an int with length of the ballot we want to predict up to.
	'threshold' is the minimum number of ballots that we need to make a prediction"""

	# Convert count to float to avoid dtype warnings
	frame['count'] = pd.to_numeric(frame['count'], errors='coerce').fillna(0.0).astype(float)

	for length in range(1, length_to_fill_to + 1):
		# Find all ballots of current length
		for row in frame.itertuples():
			if not isinstance(row.ballot, str) or len(row.ballot) != length: # catches non-str
				continue
			# We found a ballot the same length. What's the next candidate distribution for that ballot?

			# Mask for all ballots a) starting with that ballot and b) longer than it
			mask = frame['ballot'].str.startswith(row.ballot) & (frame['ballot'].str.len() > length)
			
			next_candidate_counts = {} # dict of candidate -> count
			for mask_row in frame.loc[mask, ['ballot', 'count']].itertuples():
				next_candidate = mask_row.ballot[length]  # type: ignore | zero-indexed, ballot is always a str
				if next_candidate not in next_candidate_counts:
					next_candidate_counts[next_candidate] = 0
				next_candidate_counts[next_candidate] += mask_row.count

		# If the sum of all counts is lower than threshold or nonexistent, disregard for not enough data
			if sum(next_candidate_counts.values()) < threshold or (not next_candidate_counts):
				continue

		# Else, replace the ballot with ballots extended proportionally to those counts
			for candidate, count in next_candidate_counts.items():
				new_ballot = row.ballot + candidate
				if new_ballot in frame['ballot'].values:
					frame.loc[frame['ballot'] == new_ballot, 'count'] += row.count * (count / sum(next_candidate_counts.values())) # type: ignore
				else:
					frame = pd.concat([frame, pd.DataFrame({'ballot': [new_ballot], 'count': [row.count * (count / sum(next_candidate_counts.values()))]})], ignore_index=True)
				# Drop existing ballot since we've sufficiently predicted it
			frame = frame[frame['ballot'] != row.ballot]
	return frame


In [14]:
# Create the initial dataframe
results_list = []
method = 0
i = 0
directory = 'processed_files'
subdirs = [root for root, dirs, files in os.walk(directory)][1:]
for subdir in subdirs:
	for file in os.listdir(subdir):
		if file.endswith(".pdf"):  # skip PDFs
			continue
		i += 1
		filepath = os.path.join(subdir, file)
		city_year = os.path.basename(os.path.dirname(filepath))
		_, num_candidates, num_winners, ballot_dataframe, _ = find_election_information(filepath, method)
		winners, _ = Scottish_STV(ballot_dataframe.copy(deep=True), num_candidates, num_winners, method=method)
		vote_total = int(ballot_dataframe['count'].sum())
		results_list.append({
			"file": file,
			"winners": winners,
			"num_candidates": num_candidates,
			"num_winners": num_winners,
			"vote_total": vote_total,
			"city_year": city_year
		})
		# print (f"{i}/719, {file} processed")

results_df = pd.DataFrame(results_list)

In [ ]:
# Make predictor and add predicted ballots to each election dataframe, then re-run STV to see if winners change
results_list = []
method = 0
directory = 'processed_files'
subdirs = [root for root, dirs, files in os.walk(directory)][1:]
change = 0
no_change = 0
i = 0

# Catch sets if they're stored weird, had issues with arrays of sets
def extract_set(x) -> set:
    if isinstance(x, set):
        return x
    if isinstance(x, (list, tuple, np.ndarray)) and len(x) > 0:
        return x[0] if isinstance(x[0], set) else set(x[0]) if x[0] else set()
    return set()

for subdir in subdirs:
	for file in os.listdir(subdir):
		if file.endswith(".pdf"):  # skip PDFs
			continue
		i += 1
		filepath = os.path.join(subdir, file)
		city_year = os.path.basename(os.path.dirname(filepath))
		_, num_candidates, num_winners, ballot_dataframe, _ = find_election_information(filepath, method)
		# Find the winners in results dataframe
		existing_winners = extract_set(results_df.loc[(results_df['file'] == file) & (results_df['city_year'] == city_year), 'winners'].values)

		# Predict ballots up to a certain length (e.g., 5)
		min_length = 2
		predicted_ballots = ballot_predictor(ballot_dataframe.copy(deep=True), length_to_fill_to=min_length, threshold=5)
		
		# Then run the STV with those predicted ballots
		winners, _ = Scottish_STV(predicted_ballots.copy(deep=True), num_candidates, num_winners, method=method)
		vote_total = int(ballot_dataframe['count'].sum())
		
		if winners == existing_winners:
			#print(f"{i}/719, {file} in {city_year}, no change")
			no_change += 1
		else:
			# winners have changed, reflect this
			change += 1
			print(f"{i}/719, {file} in {city_year}, winners changed from {existing_winners} to {winners}")
			results_list.append({
				"file": file,
				"city_year": city_year,
				"old_winners": existing_winners,
				"num_candidates": num_candidates,
				"num_winners": num_winners,
				"new_winners": winners,
				"vote_total": vote_total
			})
	#print (f"{i}/719, {file} processed")

print(f"{change}/{change + no_change} changed, {change/(change + no_change):.2%}")

changed_winners = pd.DataFrame(results_list)
